# Shipment Booking Prediction

Predict the next booking date and shipment type for each company based on historical booking data (2021–2025).

Baseline Approach:-

Next Booking Date - Weighted Moving Average of inter-booking gaps 

Shipment Type - Mode over a recent 90-day window 

In [36]:
import pandas as pd
import numpy as np
from datetime import timedelta

In [37]:
df = pd.read_csv('shipment_booking_data_sparse_realistic_2021_2025.csv')
df.head()

,company_name,shipment_type,booking_date
0,BlueDart,Surface,2021-01-01
1,BlueDart,Surface,2021-01-01
2,BlueDart,Surface,2021-01-01
3,BlueDart,International,2021-01-01
4,BlueDart,Surface,2021-01-01


# Exploratory Data Analysis

In [38]:
df['booking_date'] = pd.to_datetime(df['booking_date'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 128820 entries, 0 to 128819
Data columns (total 3 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   company_name   128820 non-null  str           
 1   shipment_type  128820 non-null  str           
 2   booking_date   128820 non-null  datetime64[us]
dtypes: datetime64[us](1), str(2)
memory usage: 2.9 MB


In [39]:
print(f"Total Records : {len(df):}")
print(f"Companies     : {df['company_name'].unique()}")
print(f"Shipment Types: {sorted(df['shipment_type'].unique())}")
print(f"Date Range    : {df['booking_date'].min().date()} → {df['booking_date'].max().date()}")
print(f"Missing Values: {df.isnull().sum().sum()}")

Total Records : 128820
Companies     : <StringArray>
[    'BlueDart',    'Delhivery',         'DTDC',  'FedEx India',
  'DHL Express',   'XpressBees', 'Ecom Express',    'Shadowfax']
Length: 8, dtype: str
Shipment Types: ['Air', 'Express', 'International', 'Surface']
Date Range    : 2021-01-01 → 2025-12-31
Missing Values: 0


In [40]:
#Company wise booking count
df['company_name'].value_counts()

company_name
Delhivery       33454
BlueDart        31636
DTDC            20576
XpressBees      16566
Ecom Express    12610
Shadowfax        5816
FedEx India      5415
DHL Express      2747
Name: count, dtype: int64

# Next Booking Date Prediction

Approach — Weighted Average of Inter-Arrival Gaps

1. Compute gaps (in days) between consecutive unique booking dates.
2. Use only the last 30 gaps (recent behaviour matters more).
3. Apply linearly increasing weights — the most recent gap gets the highest weight.
4. Add the rounded weighted-average gap to the last known booking date.

Weighted average used to calculate recency and avoid simple mean which treats a booking from years ago equally to yesterday.

In [41]:
def predict_next_booking_date(company_df: pd.DataFrame) -> tuple:
    unique_dates = pd.to_datetime(sorted(company_df['booking_date'].unique()))
    last_date    = unique_dates[-1]

    gaps         = pd.Series(unique_dates).diff().dt.days.dropna()
    recent_gaps  = gaps.iloc[-30:]

    weights          = np.arange(1, len(recent_gaps) + 1)
    weighted_avg_gap = np.average(recent_gaps, weights=weights)

    gap_days       = max(1, round(weighted_avg_gap))
    predicted_date = last_date + timedelta(days=gap_days)

    return predicted_date, round(weighted_avg_gap, 2)

# Shipment Type Prediction

Approach — Frequency / Mode in a Recent 90-day Window

1. Filter to the last 90 days from the company's most recent booking.
2. Count occurrences of each shipment type.
3. The type with the highest count is predicted.
4. confidence = top_type_count / total_recent_bookings

**In this type of dataset where there are less features and no strong time-series signal, frequency analysis often beats complex models on small windows.**

In [42]:
def predict_shipment_type(company_df: pd.DataFrame, last_date: pd.Timestamp) -> tuple:
    recent_cutoff = last_date - timedelta(days=90)
    recent_df     = company_df[company_df['booking_date'] >= recent_cutoff]

    if len(recent_df) == 0:
        recent_df = company_df

    type_counts = recent_df['shipment_type'].value_counts()
    total       = type_counts.sum()

    predicted_type = type_counts.index[0]
    confidence     = round(type_counts.iloc[0] / total * 100, 1)

    if len(type_counts) > 1:
        second_type = type_counts.index[1]
        second_conf = round(type_counts.iloc[1] / total * 100, 1)
    else:
        second_type = predicted_type
        second_conf = 0.0

    return predicted_type, confidence, second_type, second_conf

# Run Predictions Company Wise

In [43]:
records = []

for company in sorted(df['company_name'].unique()):
    company_df = df[df['company_name'] == company].sort_values('booking_date')
    last_date  = company_df['booking_date'].max()

    next_date, avg_gap                          = predict_next_booking_date(company_df)
    pred_type, confidence, second_type, second_conf = predict_shipment_type(company_df, last_date)

    records.append({
        'Company'                : company,
        'Last Booking Date'      : str(last_date.date()),
        'Avg Gap (days)'         : avg_gap,
        'Predicted Next Date'    : str(next_date.date()),
        'Predicted Shipment Type': pred_type,
        'Type Confidence (%)'    : confidence,
        'Alt Shipment Type'      : second_type,
        'Alt Confidence (%)'     : second_conf,
        'Total Bookings'         : len(company_df),
    })

results_df = pd.DataFrame(records)
results_df

,Company,Last Booking Date,Avg Gap (days),Predicted Next Date,Predicted Shipment Type,Type Confidence (%),Alt Shipment Type,Alt Confidence (%),Total Bookings
0,BlueDart,2025-12-31,1.20,2026-01-01,Surface,44.1,Air,29.1,31636
1,DHL Express,2025-12-31,2.39,2026-01-02,Surface,44.8,Air,31.5,2747
2,DTDC,2025-12-31,1.29,2026-01-01,Surface,43.0,Air,32.2,20576
3,Delhivery,2025-12-31,1.42,2026-01-01,Surface,45.3,Air,29.0,33454
4,Ecom Express,2025-12-31,1.45,2026-01-01,Surface,43.2,Air,30.9,12610
5,FedEx India,2025-12-26,2.46,2025-12-28,Surface,41.0,Air,34.6,5415
6,Shadowfax,2025-12-30,2.72,2026-01-02,Surface,47.8,Air,27.8,5816
7,XpressBees,2025-12-31,1.63,2026-01-02,Surface,44.3,Air,30.7,16566


In [44]:
results_df[['Company', 'Predicted Next Date', 'Predicted Shipment Type', 'Type Confidence (%)']].set_index('Company')


,Predicted Next Date,Predicted Shipment Type,Type Confidence (%)
Company,,,
BlueDart,2026-01-01,Surface,44.1
DHL Express,2026-01-02,Surface,44.8
DTDC,2026-01-01,Surface,43.0
Delhivery,2026-01-01,Surface,45.3
Ecom Express,2026-01-01,Surface,43.2
FedEx India,2025-12-28,Surface,41.0
Shadowfax,2026-01-02,Surface,47.8
XpressBees,2026-01-02,Surface,44.3


In [45]:
results_df.to_csv('initial_time-series_shipment_predictions_output.csv', index=False)

# Machine Learning Approach — Random Forest

Approach 2

After the baseline statistical approach tried to use random forest classifier to predict shipment type.

The goal is to evaluate whether a machine learning model could capture additional patterns beyond simple frequency-based logic.

In [59]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

In [47]:
df = pd.read_csv("shipment_booking_data_sparse_realistic_2021_2025.csv")
df['booking_date'] = pd.to_datetime(df['booking_date'])
df = df.sort_values(['company_name', 'booking_date'])

# Feature Engineering
features to train the model:

- Day of Week
- Month
- Previous Shipment Type (Lag Feature)
- Company Encoding:

In [48]:
df['day_of_week'] = df['booking_date'].dt.dayofweek
df['month'] = df['booking_date'].dt.month

# Lag feature
df['prev_shipment'] = df.groupby('company_name')['shipment_type'].shift(1)

df = df.dropna()

In [49]:
df['shipment_type_encoded'] = df['shipment_type'].astype('category').cat.codes
df['prev_shipment_encoded'] = df['prev_shipment'].astype('category').cat.codes
df['company_encoded'] = df['company_name'].astype('category').cat.codes

# Train-Test Split

In [50]:
features = ['day_of_week', 'month', 'prev_shipment_encoded', 'company_encoded']
target = 'shipment_type_encoded'

X = df[features]
y = df[target]

split_index = int(len(df) * 0.8)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

# Model Training

In [51]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Random Forest Accuracy:", round(accuracy * 100, 2))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Random Forest Accuracy: 38.17 %

Classification Report:
               precision    recall  f1-score   support

           0       0.31      0.23      0.26      7742
           1       0.21      0.15      0.17      5215
           2       0.14      0.01      0.01      1301
           3       0.45      0.63      0.52     11505

    accuracy                           0.38     25763
   macro avg       0.28      0.25      0.24     25763
weighted avg       0.34      0.38      0.35     25763



In [55]:
import warnings
warnings.filterwarnings('ignore')

In [57]:
rf_results = []

shipment_mapping = dict(enumerate(df['shipment_type'].astype('category').cat.categories))

for company in df['company_name'].unique():
    temp = df[df['company_name'] == company].sort_values('booking_date')
    
    last_row = temp.iloc[-1]
    last_date = last_row['booking_date']
    
    input_features = [[
        last_row['day_of_week'],
        last_row['month'],
        last_row['prev_shipment_encoded'],
        last_row['company_encoded']
    ]]
    
    pred_encoded = model.predict(input_features)[0]
    pred_shipment = shipment_mapping[pred_encoded]
    

    unique_dates = pd.to_datetime(sorted(temp['booking_date'].unique()))
    gaps = pd.Series(unique_dates).diff().dt.days.dropna()
    recent_gaps = gaps.iloc[-30:]
    
    if len(recent_gaps) > 0:
        weights = np.arange(1, len(recent_gaps) + 1)
        weighted_avg_gap = np.average(recent_gaps, weights=weights)
        gap_days = max(1, round(weighted_avg_gap))
    else:
        gap_days = 1
    
    next_date = last_date + pd.Timedelta(days=gap_days)
    
    rf_results.append([
        company,
        next_date,
        pred_shipment
    ])

rf_df = pd.DataFrame(rf_results, columns=[
    'Company',
    'RF_Predicted_Next_Date',
    'RF_Predicted_Shipment_Type'
])


rf_df



,Company,RF_Predicted_Next_Date,RF_Predicted_Shipment_Type
0,BlueDart,2026-01-01,Surface
1,DHL Express,2026-01-02,Air
2,DTDC,2026-01-01,Surface
3,Delhivery,2026-01-01,Surface
4,Ecom Express,2026-01-01,Surface
5,FedEx India,2025-12-28,Express
6,Shadowfax,2026-01-02,Express
7,XpressBees,2026-01-02,Express


# Observations

- The model predictions were heavily biased toward the most frequent shipment type (Surface).
- The model struggled to capture meaningful patterns beyond dominant frequency trends.

This indicates that the dataset lacks strong predictive signals for classification.

In [58]:
rf_df.to_csv("Random_Forest_shipment_predictions_output.csv", index=False)

# Comparison: Baseline vs Random Forest

- Shipment Type Predictions:
  - High overlap between both approaches
  - Majority predictions remained the same (Surface)

- Next Booking Date:
  - Nearly identical, as both relied on gap-based logic

- Overall Insight:
  - Machine learning did not significantly outperform the baseline approach

# Final Conclusion

**Given the limited feature space and dominant frequency patterns, a statistical approach provides similar performance with greater interpretability and efficiency compared to machine learning models.**

If richer data were available, model performance could be improved by incorporating:

- Shipment volume / order size  
- Geographic information  
- Customer data

This would enable machine learning models to capture deeper patterns and outperform statistical methods.
